In [5]:
import json
import statistics
import os
import pandas as pd
from typing import List, Dict, Any, Tuple, Optional

def process_file(file_path: str) -> Tuple[Optional[float], Optional[float]]:
    """
    Lit un fichier JSON et retourne (pourcentage_strong, mean_violation).
    Retourne (None, None) si le fichier n'existe pas ou est vide/invalide.
    """
    violations: List[float] = []
    number_of_strong: int = 0

    try:
        if not os.path.exists(file_path):
            return None, None
            
        with open(file_path, 'r', encoding='utf-8') as f:
            data: List[Dict[str, Any]] = json.load(f)

        for item in data:
            violation_metric: float = item.get("violation_metric")
            is_strong: bool = item.get("strong", False)
            
            if violation_metric is not None:
                violations.append(violation_metric)
            if is_strong:
                number_of_strong += 1
        
        if not violations:
            return None, None

        mean_violation = statistics.mean(violations)
        percentage_strong = 100 * number_of_strong / len(violations)
        
        return percentage_strong, mean_violation

    except Exception as e:
        # On peut décommenter le print pour le debug si besoin
        # print(f"Erreur lors du traitement de {file_path}: {e}")
        return None, None

def generate_study_results_table(
    results_dir: str = "results", 
    csv_path: str = "type-experience.csv"
) -> pd.DataFrame:
    """
    Génère un DataFrame Pandas résumant les résultats de l'étude en croisant
    les configurations définies dans le CSV avec les fichiers présents dans le dossier results.
    """
    
    # 1. Chargement de la configuration des expériences
    try:
        df_config = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Erreur lors de la lecture du CSV '{csv_path}': {e}")
        return pd.DataFrame()

    # 2. Définition des catégories et des mots-clés pour les trouver dans les noms de fichiers
    # Le format est "Nom Colonne": ["mot_cle_1", "mot_cle_2"] (pour gérer les variations comme monotonicity/monotonic)
    categories_keywords = {
        "Negation": ["negated"],
        "Paraphrasing": ["paraphrase"],
        "Monotonicity": ["monotonic"], # Trouvera "monotonic" et "monotonicity"
        "Bayes' rule": ["bayes"]
    }

    rows = []
    
    # Liste de tous les fichiers disponibles pour éviter de faire des os.path.join à l'aveugle
    try:
        available_files = os.listdir(results_dir)
    except FileNotFoundError:
        print(f"Le dossier '{results_dir}' n'existe pas.")
        return pd.DataFrame()

    # 3. Itération sur chaque configuration d'expérience du CSV
    for _, row in df_config.iterrows():
        model_name = str(row['Modèle LLM']).strip()
        temp = str(row['Température'])
        response_format = str(row['Format de Réponse (indiqué dans le prompt)']).strip()
        nb_exp = str(row['Nb exp par questions'])
        
        # Création du nom d'affichage pour le modèle (ex: "gpt-4 (T=0.8, long)")
        display_name = f"{model_name} (T={temp}, {response_format})"
        row_data = {"Experience Params (model, temperature T, reasoning)": display_name}
        
        # Nettoyage du nom du modèle pour la correspondance de fichier
        # "deepseek/deepseek-chat" devient souvent "deepseek-deepseek-chat" dans les noms de fichiers
        model_search_term = model_name.replace("/", "-")
        
        # Pour chaque catégorie (colonne)
        for cat_name, keywords in categories_keywords.items():
            
            target_file = None
            
            # Recherche du fichier correspondant dans le dossier
            for filename in available_files:
                # Vérification des critères : Modèle, Température, Format, Nombre d'expériences, Catégorie
                # On utilise des vérifications souples (in) pour gérer les préfixes/suffixes variables
                if (
                    model_search_term in filename and
                    f"T_{temp}" in filename and
                    f"_{response_format}.json" in filename and
                    f"times_{nb_exp}" in filename and
                    any(k in filename for k in keywords)
                ):
                    target_file = filename
                    break
            
            # Calcul des stats si le fichier est trouvé
            col_pct = f"{cat_name} (>0.2)"
            col_mean = f"{cat_name} (Mean)"
            
            if target_file:
                full_path = os.path.join(results_dir, target_file)
                pct_strong, mean_val = process_file(full_path)
                
                if pct_strong is not None:
                    row_data[col_pct] = f"{pct_strong:.1f}%"
                    row_data[col_mean] = f"{mean_val:.2f}"
                else:
                    row_data[col_pct] = "N/A"
                    row_data[col_mean] = "N/A"
            else:
                # Fichier non trouvé
                row_data[col_pct] = "-"
                row_data[col_mean] = "-"
        
        rows.append(row_data)

    # 4. Création et mise en forme du DataFrame
    df = pd.DataFrame(rows)
    
    # Réorganiser les colonnes
    desired_order = ["Experience Params (model, temperature T, reasoning)"]
    for cat in categories_keywords.keys():
        desired_order.append(f"{cat} (>0.2)")
        desired_order.append(f"{cat} (Mean)")
        
    # On s'assure que les colonnes existent (au cas où le CSV serait vide)
    existing_cols = [c for c in desired_order if c in df.columns]
    
    return df[existing_cols]

# Exemple d'utilisation :
df_final = generate_study_results_table(results_dir="results", csv_path="type-experience.csv")
display(df_final)

,"Experience Params (model, temperature T, reasoning)",Negation (>0.2),Negation (Mean),Paraphrasing (>0.2),Paraphrasing (Mean),Monotonicity (>0.2),Monotonicity (Mean),Bayes' rule (>0.2),Bayes' rule (Mean)
0,"deepseek/deepseek-chat-v3.1 (T=0.0, long)",12.5%,0.07,75.0%,0.44,14.3%,0.10,100.0%,0.54
1,"meta-llama/llama-3.1-8b-instruct (T=0.0, long)",25.0%,0.13,62.5%,0.34,0.0%,0.16,75.0%,0.31
2,"anthropic/claude-3-haiku (T=0.0, long)",25.0%,0.15,62.5%,0.27,100.0%,0.35,50.0%,0.19
3,"gpt-4 (T=0.8, long)",37.5%,0.20,37.5%,0.15,0.0%,0.02,75.0%,0.40
4,"gpt-4 (T=0.3, long)",50.0%,0.32,37.5%,0.21,28.6%,0.23,87.5%,0.39
5,"gpt-4 (T=0.0, short)",14.3%,0.07,37.5%,0.17,N/A,N/A,71.4%,0.31
6,"gpt-4 (T=0.5, short)",0.0%,0.10,12.5%,0.10,N/A,N/A,66.7%,0.28


### Resultats de l'étude (tirés du papier)

| Model | Negation (>0.2) | Negation (Mean) | Paraphrasing (>0.2) | Paraphrasing (Mean) | Monotonicity (>0.2) | Monotonicity (Mean) | Bayes’ rule (>0.2) | Bayes’ rule (Mean) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| GPT-3.5-turbo (temp=0) | 52.6% | 0.34 | 30.8% | 0.21 | 42.0% | 0.23 | 68.6% | 0.28 |
| GPT-3.5-turbo (temp=0.5) | 58.9% | 0.31 | 22.1% | 0.16 | 26.0% | 0.14 | 64.7% | 0.24 |
| GPT-4 (temp=0) | 10.9% | 0.10 | 12.5% | 0.13 | 16.0% | 0.11 | 58.8% | 0.25 |
| GPT-4 (temp=0.5) | 8.6% | 0.09 | 14.4% | 0.13 | 12.0% | 0.09 | 74.5% | 0.27 |

Table: Mean violation magnitude and fraction of “strong” violations (with value above ε = 0.2)

## Analyse

Chaque expérience a été menée avec 8 questions (ou groupe de questions) différentes tirées de l'étude originale. Ce choix, qui est moins élevé que le nombre de questions utilisées par l'étude originale, repose en partie sur la quantité d'argent dont nous disposons pour interroger les llms avec openrouter. Il faut donc bien prendre en compte que nos analyses se basent que sur un corpus de questions réduit.

### Variation du modèle de LLM

Pour ces expériences (0,1,2), nous avons choisi de faire varier le LLM choisi afin de voir si celui-ci a un impact sur les performances de prédiction obtenue. 

Les LLMs que nous avons utilisé présentent des caractéristiques différentes. DeepSeek est un modèle orienté raisonnement qui est peu cher, llama est un modèle largement utilisé dans l'industrie et enfin le modèle claude est quant à lui le plus cher et nous voulons voir si ce prix plus élevé induit de meilleurs performances.

DeepSeek montre une excellente robustesse sur la Négation et sur la monotonicité de même que pour le modèle Llama.
Claude-3 présente quant à lui des résultats moins bons sur la monotonicité mais présente de bon résultats avec l'expérience sur la loi de Bayes. 

Globalement, nous pouvons conclure que chaque modèle présente des résultats qui seront différents. Ainsi le choix du modèle à un impact non négligeable sur la qualité des réponses. Certains modèle (les moins performants comme gpt-3.5 turbo) peuvent présenter des résulats moins bons mais nous constatons aussi que la majorité des modèles éprouvent un difficulté à satisfaire la loi de Bayes. Le "meilleur" modèle étant Claude avec un taux d'erreur de 0.2. 

### Variation de la température

Pour ces expériences (3,4), nous avons choisi de faire varier la température du LLM pour un modèle fixé : gpt-4 (utilisé dans l'étude principale) afin de voir si cellle-ci a un impact sur les performances de prédiction obtenue.

Contre-intuitivement, la température plus élevée (0.8) a produit de meilleurs résultats (moins de violations) que la température plus basse (0.3) sur la Négation et surtout la Monotonicité (0% vs 28.6% de violations fortes).

Cela suggère qu'une température plus élevée peut parfois aider le modèle à "sortir" de mauvaises ornières de raisonnement logique sur des tâches complexes. Cependant, nous remarquons qu'avoir une température plus élevée produit des résultats globalements moins bons comparés aux autres résultats utilisants d'autres modèles sur la Négation alors que cette expérience est celle qui présente les meilleurs résultats au global. 

Nous pouvons donc conclure que la température à une influence assez forte sur la qualité du résultat produit et qu'elle peut dans certains cas aider le llm à se donner de la liberté dans son raisonnement.

### Format de la réponse produite (influence sur le raisonnement)

Pour ces expériences (5,6), nous avons choisi de faire varier le prompt système donné au LLM afin de faire varier la réponse. Au lieu de nous fournir son raisonnement en plus de la valeur estimée, nous lui demandons seulement la valeur. Cela a pour but d'essayer de limiter son raisonnement pour un modèle fixé : gpt-4 (utilisé dans l'étude principale) afin de voir si cellle-ci a un impact sur les performances de prédiction obtenue. Intuitivement, nous pouvons supposer que les performances de prédictions seront moins bonnes.

Toutefois, le passage au format court semble bénéfique (résultats très solides pour la Négation, 0% de violations fortes à T=0.5). Le format court semble aussi réduire la fréquence des violations sur le Paraphrasing et la Négation. Ces résultats sont légèrements contre intuitifs car nous ne demandons pas au llm de développer son raisonnement et pourtant celui-ci arrive à produire de bons résultats. 

Nous remarquons cependant que le LLM n'a pas réussi à produire de résultats pour l'expérience avec la Monotonicité. Ceci n'est pas un oubli de notre part ni un problème dans le code, le llm ne parvient pas à produire de réponse et nous indique plutôt qu'étant une IA, il ne peut produire des prédictions d'évenements car cela est contraire au son utilisation.

## Conclusion

Nous concluons que le résultat de l'étude, qui est de conclure que les LLMs sont de mauvais oracles de prédictions sur les évenements futurs, est toujours vérifiés. Pour cela nous avons appliqué les même mesures de violations et obtenus des résultats qui ne nous permettent pas de dire qu'une certaine configuration (modèle, température, raisonnement, ...) permet d'obtenir des prédictions cohérentes et fiables.